In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
def generate_sequence(seq_len):
    x = np.random.randint(0, 10, size=seq_len, dtype=np.int64)
    first = x[0]
    y = (x + first) % 10
    return x, y

In [3]:
x, y = generate_sequence(10)
print('x:', x)
print('y:', y)

x: [9 2 1 6 1 4 7 8 2 5]
y: [8 1 0 5 0 3 6 7 1 4]


In [4]:
class DigitSequenceDataset(Dataset):
    def __init__(self, num_samples, seq_len):
        self.x_data = []
        self.y_data = []

        for _ in range(num_samples):
            x, y = generate_sequence(seq_len)
            self.x_data.append(x)
            self.y_data.append(y)

        self.x_data = torch.tensor(np.array(self.x_data), dtype=torch.long)
        self.y_data = torch.tensor(np.array(self.y_data), dtype=torch.long)

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

In [5]:
seq_len = 20
train_dataset = DigitSequenceDataset(num_samples=10000, seq_len=seq_len)
test_dataset = DigitSequenceDataset(num_samples=2000, seq_len=seq_len)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
class SeqModel(nn.Module):
    def __init__(self, model_type='RNN', embed_dim=16, hidden_dim=32, num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=10, embedding_dim=embed_dim)

        if model_type == 'RNN':
            self.rnn = nn.RNN(
                input_size=embed_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True
            )
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(
                input_size=embed_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True
            )
        elif model_type == 'GRU':
            self.rnn = nn.GRU(
                input_size=embed_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True
            )
        else:
            raise ValueError('model_type must be one of: RNN, LSTM, GRU')

        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.rnn(emb)
        logits = self.fc(out)
        return logits

In [7]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

device

'mps'

In [8]:
def evaluate_model(model, data_loader, criterion=None):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)

            if criterion is not None:
                loss = criterion(
                    logits.reshape(-1, 10),
                    y_batch.reshape(-1)
                )
                total_loss += loss.item()

            preds = logits.argmax(dim=-1)
            total_correct += (preds == y_batch).sum().item()
            total_count += y_batch.numel()

    avg_loss = total_loss / len(data_loader) if criterion is not None else None
    acc = total_correct / total_count
    return avg_loss, acc

In [9]:
def train_model(model, train_loader, test_loader, epochs=10, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(x_batch)  # [B, T, 10]

            loss = criterion(
                logits.reshape(-1, 10),
                y_batch.reshape(-1)
            )

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            preds = logits.argmax(dim=-1)
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.numel()

        train_acc = train_correct / train_total

        test_loss, test_acc = evaluate_model(model, test_loader, criterion)

        print(
            f'Epoch {epoch:02d} | '
            f'train_loss={train_loss / len(train_loader):.4f} | '
            f'train_acc={train_acc:.4f} | '
            f'test_loss={test_loss:.4f} | '
            f'test_acc={test_acc:.4f}'
        )
    return model

In [10]:
rnn_model = SeqModel(model_type='RNN', embed_dim=16, hidden_dim=32)
rnn_model = train_model(rnn_model, train_loader, test_loader, epochs=10, lr=1e-3)

Epoch 01 | train_loss=2.2955 | train_acc=0.1378 | test_loss=2.2862 | test_acc=0.1447
Epoch 02 | train_loss=2.2770 | train_acc=0.1448 | test_loss=2.2643 | test_acc=0.1459
Epoch 03 | train_loss=2.2472 | train_acc=0.1453 | test_loss=2.2295 | test_acc=0.1458
Epoch 04 | train_loss=2.2155 | train_acc=0.1439 | test_loss=2.2045 | test_acc=0.1466
Epoch 05 | train_loss=2.1988 | train_acc=0.1459 | test_loss=2.1942 | test_acc=0.1436
Epoch 06 | train_loss=2.1896 | train_acc=0.1452 | test_loss=2.1867 | test_acc=0.1441
Epoch 07 | train_loss=2.1823 | train_acc=0.1442 | test_loss=2.1787 | test_acc=0.1467
Epoch 08 | train_loss=2.1695 | train_acc=0.1489 | test_loss=2.1340 | test_acc=0.1593
Epoch 09 | train_loss=1.8743 | train_acc=0.2559 | test_loss=1.6740 | test_acc=0.3101
Epoch 10 | train_loss=1.5280 | train_acc=0.3388 | test_loss=1.4072 | test_acc=0.3473


In [11]:
lstm_model = SeqModel(model_type='LSTM', embed_dim=16, hidden_dim=32)
lstm_model = train_model(lstm_model, train_loader, test_loader, epochs=10, lr=1e-3)

Epoch 01 | train_loss=2.2993 | train_acc=0.1291 | test_loss=2.2941 | test_acc=0.1447
Epoch 02 | train_loss=2.2805 | train_acc=0.1500 | test_loss=2.2363 | test_acc=0.1800
Epoch 03 | train_loss=1.8585 | train_acc=0.3480 | test_loss=1.4192 | test_acc=0.4535
Epoch 04 | train_loss=1.1965 | train_acc=0.5245 | test_loss=1.0078 | test_acc=0.6065
Epoch 05 | train_loss=0.8611 | train_acc=0.6888 | test_loss=0.7131 | test_acc=0.7740
Epoch 06 | train_loss=0.5763 | train_acc=0.8561 | test_loss=0.4412 | test_acc=0.9304
Epoch 07 | train_loss=0.3422 | train_acc=0.9675 | test_loss=0.2506 | test_acc=0.9934
Epoch 08 | train_loss=0.1925 | train_acc=0.9967 | test_loss=0.1466 | test_acc=0.9990
Epoch 09 | train_loss=0.1189 | train_acc=0.9994 | test_loss=0.0956 | test_acc=0.9997
Epoch 10 | train_loss=0.0824 | train_acc=0.9997 | test_loss=0.0731 | test_acc=0.9988


In [12]:
gru_model = SeqModel(model_type='GRU', embed_dim=16, hidden_dim=32)
gru_model = train_model(gru_model, train_loader, test_loader, epochs=10, lr=1e-3)

Epoch 01 | train_loss=2.2994 | train_acc=0.1276 | test_loss=2.2929 | test_acc=0.1443
Epoch 02 | train_loss=2.2852 | train_acc=0.1461 | test_loss=2.2751 | test_acc=0.1447
Epoch 03 | train_loss=2.2117 | train_acc=0.1823 | test_loss=2.0484 | test_acc=0.2615
Epoch 04 | train_loss=1.7352 | train_acc=0.3902 | test_loss=1.3774 | test_acc=0.5300
Epoch 05 | train_loss=1.0813 | train_acc=0.6874 | test_loss=0.8215 | test_acc=0.8264
Epoch 06 | train_loss=0.6447 | train_acc=0.8938 | test_loss=0.4949 | test_acc=0.9432
Epoch 07 | train_loss=0.3969 | train_acc=0.9682 | test_loss=0.3122 | test_acc=0.9853
Epoch 08 | train_loss=0.2562 | train_acc=0.9940 | test_loss=0.2075 | test_acc=0.9972
Epoch 09 | train_loss=0.1733 | train_acc=0.9984 | test_loss=0.1427 | test_acc=0.9997
Epoch 10 | train_loss=0.1205 | train_acc=0.9999 | test_loss=0.1026 | test_acc=1.0000


In [13]:
def predict_sequence(model, x_seq):
    model.eval()

    x_tensor = torch.from_numpy(x_seq).long().unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x_tensor)
        preds = logits.argmax(dim=-1).squeeze(0).cpu().numpy()

    return preds

In [14]:
x, y_true = generate_sequence(seq_len=10)
y_pred = predict_sequence(gru_model, x)

print('x      :', x)
print('y_true :', y_true)
print('y_pred :', y_pred)

x      : [6 7 3 1 2 9 0 2 2 9]
y_true : [2 3 9 7 8 5 6 8 8 5]
y_pred : [2 3 9 7 8 5 6 8 8 5]
